# SplatStream Lab — Reproducible CUDA Evaluation

This notebook reproduces the real-scene validation path for SplatStream Lab using **Mip-NeRF 360 / Bonsai** and a pinned `gsplat` revision.

The first stage records the software/hardware environment, trains a 7,000-step 3D Gaussian Splatting baseline, exports the resulting checkpoint to PLY, and executes the project's portable compression sanity-check pipeline.

The second stage reuses the trained checkpoint to run a **full-scene held-out CUDA rate-distortion sweep** across pruning and quantization operating points.

**Runtime requirement:** select a GPU runtime before execution.


## 1. Train and validate the 7K Bonsai baseline


In [ ]:
from pathlib import Path
import urllib.request

runner = Path('/content/run_bonsai_cuda.py')
url = 'https://raw.githubusercontent.com/reusahn/splatstream-lab/main/tools/run_bonsai_cuda.py'
urllib.request.urlretrieve(url, runner)
print('Runner:', runner)

%run /content/run_bonsai_cuda.py


## 2. Full-scene CUDA rate-distortion sweep

This stage checks whether the pinned source tree and 7,000-step checkpoint are still present in the active Colab runtime. If the runtime was reset and `/content` was cleared, it automatically rebuilds the missing baseline state before starting the sweep.

The evaluator applies the same view-independent importance heuristic to the full scene, creates four pruning/quantization operating points, and evaluates every point on the same held-out Bonsai cameras with the full `gsplat` rasterizer. It keeps one `gsplat` Runner, dataset, CUDA stack, and metric networks alive for the entire sweep.


In [ ]:
from pathlib import Path
import urllib.request

pipeline = Path('/content/run_full_cuda_pipeline.py')
url = 'https://raw.githubusercontent.com/reusahn/splatstream-lab/main/tools/run_full_cuda_pipeline.py'
urllib.request.urlretrieve(url, pipeline)
print('Full CUDA pipeline:', pipeline)

%run /content/run_full_cuda_pipeline.py


## Outputs

The baseline stage produces **`splatstream_bonsai_evidence.zip`**. The full CUDA sweep produces **`splatstream_full_cuda_rd_v2.zip`**, containing rate-distortion CSV/JSON, method notes, environment metadata, plots, and validation outputs.

Measured values should be reported together with the GPU model, pinned commit IDs, dataset source, evaluation split, and payload definition.
